# Combine data: HK1-R12Ter
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from itertools import product

import pandas as pd

from hk1_r12ter_analysis.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR, logger
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Combine normalized data

In [ ]:
sample_key = "Sample"
group_key = "Group"
source_types = ["RBC", "Plasma"]

value_type = "Intensities"

normalization_inputs = (
    # Sample Normalization, Data Transformation, and Data Scaling
    [None, None, None],
    ["median", None, None],
    ["median", None, "auto"],
)
normalization_strs = [
    "-".join([str(val).lower() for val in values]) for values in normalization_inputs
]

source_types = ["RBC", "Plasma"]
combinations_data_types = {
    # Combined data type : (Data types, starting index for types to keep)
    "MetabolitesOxylipins": (["Metabolites", "Oxylipins"], 0),
    "CombinedLipids": (["Metabolites", "Lipids", "Oxylipins"], 1),
    "CombinedMetabolomics": (["Metabolites", "Lipids", "Oxylipins"], 0),
    "All": (["Metabolites", "Lipids", "Oxylipins", "Proteins"], 0),
}
# Directories
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized"
output_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized"

In [ ]:
normalization_strs = [
    "-".join([str(val).lower() for val in values]) for values in normalization_inputs
]
index_cols = [key for key in [sample_key, group_key] if key]
for norm_str in normalization_strs:
    # Set directory paths
    input_dirpath_norm = input_dirpath / norm_str
    output_dirpath_norm = output_dirpath / norm_str
    output_dirpath_norm.mkdir(parents=True, exist_ok=True)
    logger.debug(
        "Combining data using sample normalization: {}; data transformation: {}; data scaling: {}.".format(
            *norm_str.split("-")
        )
    )

    # Iterate through combinations
    for source_type, (combination_data_type, (data_types, idx_to_keep)) in product(
        source_types, combinations_data_types.items()
    ):
        logger.debug(f"Processing data for '{source_type}-{combination_data_type}'...")
        # Load data and add to list for merging
        logger.debug(f"Combining data for '{source_type}-{combination_data_type}'...")
        to_merge = []
        for data_type in data_types:
            logger.debug(f"Adding '{source_type}-{data_type}' data for combining...")
            filename = make_filename(source_type, data_type, value_type)
            df = load_dataframe_from_file(input_dirpath_norm / filename, index_col=index_cols)
            df.columns = pd.MultiIndex.from_tuples([(data_type, col) for col in df.columns])
            to_merge += [df]
            logger.info(f"Added '{source_type}-{data_type}' data for combining.")
        logger.debug("Merging data....")
        df = to_merge[0].join(to_merge[1:], how="outer")
        # Remove duplicates, keeping only those added last
        logger.debug("Removing duplicates...")
        duplicated = df.columns.get_level_values(level=1).duplicated(keep="last")
        df = df.loc[:, pd.IndexSlice[:, ~duplicated]]
        logger.info(f"Number of duplicates removed: {duplicated.astype(int).sum()}")
        # Keep only relevant data types
        logger.debug(f"Keeping {data_types[idx_to_keep:]} data, removing other types...")
        df = df.loc[:, data_types[idx_to_keep:]]
        df = df.droplevel(0, axis=1)
        logger.info(f"Combined data for '{source_type}-{combination_data_type}'.")
        #  Save combined data
        filename = make_filename(source_type, combination_data_type, value_type)
        save_dataframe_to_file(df, output_dirpath_norm / filename, index=True)
        logger.info(f"Processed data for '{source_type}-{combination_data_type}'.")
    logger.info(
        "Combined data for sample normalization: {}; data transformation: {}; data scaling: {}.".format(
            *norm_str.split("-")
        )
    )